# **Install necessary libraries**

In [ ]:
# Unzip the library
!unzip unsloth.zip

# Go inside the directory
%cd /content/unsloth

# Install unsloth library with the configuration
!pip install -e ".[cu128-torch280]"

# Install flash-attention-2
!pip install flash-attn --no-build-isolation --no-cache-dir \
    --find-links https://github.com/Dao-AILab/flash-attention/releases/tag/v2.8.3

# **Initialize Weights and Biases (WANDB)**

In [ ]:
# Google Colab way of setting environment variables
import wandb
import os
from google.colab import userdata

# Inject api key & project to `os.environ`
os.environ['HF_TOKEN'] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = userdata.get("WANDB_PROJECT")

if wandb.login():
    print("Successfully logged !")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: tahar-masmaliyev (tahar-masmaliyev-george-washington-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged !


In [ ]:
# Default environment variables setting
import wandb
import os

if wandb.login():
    print("Successfully logged !")

# **Load Libraries**

In [1]:
import torch
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer

from datasets import (load_dataset, concatenate_datasets,
                      Dataset, DatasetDict)

from tqdm.notebook import tqdm
from typing import Dict, List, Optional, Any

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# **Initialize Model**

In [2]:
# Configure model settings
model_name = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 20000
LORA_RANK = 16

In [ ]:
# Load the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False,
    attn_implementation="flash_attention_2",
    dtype=torch.bfloat16,
)

# Define & add special tokens to tokenizer
format_tag = [
    '<ANSWER>',   '</ANSWER>',
    '<CODE>',     '</CODE>',
    '<COT>',      '</COT>',
    '<LONG_COT>', '</LONG_COT>'
]
tokenizer.add_tokens(format_tag, special_tokens=True)

# Resize the embedding size to match tokenizer length
# model.resize_token_embeddings(len(tokenizer))

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce GTX 1650. Num GPUs = 1. Max memory: 3.628 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[accelerate.big_modeling|WARNING]Some parameters are on the meta device because they were offloaded to the cpu.


unsloth/Qwen3-4B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


7

In [ ]:
# Apply PEFT (LORA) wrapper
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"
    ],
    lora_alpha = LORA_RANK * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
# Or use the model's built-in method
model.base_model.model.tie_weights()

# **Dataset Loader & Tokenizer**

In [10]:
def tokenize(example: Dict[str, str]) -> Dict[str, List[int]]:
    # Get Instruction & Solution pairs
    instruction = example['instruction']
    solution = example['output']

    # Define prompt message
    prompt_message = [
        {'role': 'user',      'content': instruction}
    ]

    # Tokenize only prompt message
    prompt_ids = tokenizer.apply_chat_template(
        prompt_message,
        tokenize=True,
        add_generation_prompt=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )['input_ids']

    # Get length of tokenized prompt message
    prompt_length = len(prompt_ids)

    # Define full conversation
    full_message = [
        {'role': 'user',      'content': instruction},
        {'role': 'assistant', 'content': solution}
    ]

    # Tokenize full conversation
    input_ids = tokenizer.apply_chat_template(
        full_message,
        tokenize=True,
        add_generation_prompt=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )['input_ids']

    # Set `ignore_index` to prompt tokens to ignore during loss calculation
    labels = [-100] * prompt_length + input_ids[prompt_length:]

    # Return dictionary with corresponding fields
    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
        'labels': labels
    }

In [11]:
# Load the train& test datasets
ds_train = load_dataset('taharmasmaliyev07/arm-sft-stage-dataset', split='train')
ds_val = load_dataset('taharmasmaliyev07/arm-sft-stage-dataset', split='test')

# Tokenize the data
dst_train = ds_train.map(tokenize, remove_columns=ds_train.column_names)
dst_val = ds_val.map(tokenize, remove_columns=ds_val.column_names)

Map:   0%|          | 0/4331 [00:00<?, ? examples/s]

# **Model training**

In [ ]:
# Model training configuration
EPOCH = 3

USE_BF16 = True
BF_TAG = 'b16' if USE_BF16 else 'f16'

In [ ]:
# Initialize training configuration
training_args = SFTConfig(
    # Main Configuration
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 4,
    num_train_epochs = EPOCH,
    
    # Training algorithm
    warmup_ratio = 0.1,
    weight_decay = 0.01,
    optim = 'adamw_8bit',
    bf16 = USE_BF16,

    # LeMAX_SEQ_LENGTHarning Rate & Scheduler
    learning_rate = 2e-4,
    lr_scheduler_type = 'cosine',

    # Steps to write to
    logging_steps = 50,
    eval_steps = 50,

    # Metrics to look at
    save_strategy = 'no',
    eval_strategy = 'steps',
    metric_for_best_model = 'eval_loss',
    
    # Checkpoint & Is greater or less
    load_best_model_at_end = False,
    greater_is_better = False,

    # Reporting
    report_to = 'wandb',
    run_name=f"{model_name}-{BF_TAG}-epoch-{EPOCH}",
)

# SFT- Trainer initialization
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    args = training_args,
    train_dataset = dst_train,
    eval_dataset = dst_val,
)

# Start training process
trainer.train()

# **Push model parameters to HuggingFace**

In [ ]:
# Push model+lora adapters as whole to huggingface hub
model.push_to_hub_merged(
    "taharmasmaliyev07/Qwen-3-8B-b16-tuned-full",
    tokenizer,
    token = os.environ['HF_TOKEN']
)

# Push only lora adapters to huggingface hub
model.push_to_hub(
    "taharmasmaliyev07/Qwen-3-8B-b16-tuned-adapters",
    token = os.environ['HF_TOKEN']
)